Ablations for phase 3

# Citations: 
1) LoRMA: Low-Rank Multiplicative Adaptation for LLMs, Bihany et. al.
2) LoRA: Low-Rank Adaptation of Large Language Models Hu et. al.
3) AdaLoRA: Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning Zhang et. al.
4) 



In [ ]:
!pip install ijson
import os
import sys
import json
import math
import time
import random
import zipfile
from typing import List, Tuple, Dict, Any, Generator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import numpy as np, ijson
from multiprocessing import Pool
import regex
!pip install -q peft scikit-learn pandas
!pip uninstall -y torchao

In [ ]:

HF_TOKEN = None
if "HF_TOKEN" in os.environ:
    HF_TOKEN = os.environ["HF_TOKEN"]
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF")
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("Successfully loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        print(f"Note: Kaggle secrets not detected or HF key missing ({e}). Will attempt public access.")

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Huggingface hub authentication successful.")
    except Exception as err:
        print(f"HF Login warning: {err}")


print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# label vocab: 
'''{
    "Fact": 0,
    "Issue": 1,
    "ArgumentPetitioner": 2,
    "ArgumentRespondent": 3,
    "Statute": 4,
    "Dissent": 5,
    "PrecedentReliedUpon": 6,
    "PrecedentNotReliedUpon": 7,
    "PrecedentOverruled": 8,
    "RulingByLowerCourt": 9,
    "RatioOfTheDecision": 10,
    "RulingByPresentCourt": 11,
    "None": 12
}'''

In [ ]:
from huggingface_hub import HfApi

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

HF_TOKENIZER_UPLOAD_REPO = os.environ.get("HF_TOKENIZER_UPLOAD_REPO", "legal-bpe-tokenizer-phase1")



TRAIN_FILES = ["Phase_3_RR/Train/CL_dev.json", "Phase_3_RR/Train/CL_train.json",
              "Phase_3_RR/Train/IT_dev.json", "Phase_3_RR/Train/IT_train.json"]

TEST_FILES = ["Phase_3_RR/Test/CL_test.json", "Phase_3_RR/Test/IT_test.json"]

VOCAB_SIZE = 33000
SEED = 67
random.seed(SEED)
torch.manual_seed(SEED)

KNOWN_COUNTS = {"train_supreme_court.json": 55549, "train_acts.json": 18826,
                "train_high_courts.json": 2536738, "train_tribunals.json": 869132}

HF_TOK_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-bpe-tokenizer-phase2-v1"

OUTPUT_DIR = "/kaggle/working/cs781_phase3" if os.path.exists("/kaggle") else "./outputs"
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "custom_bpe_tokenizer_v2")
TOKEN_DATA_DIR = os.path.join(OUTPUT_DIR, "token_data_v3")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "model_checkpoint")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(TOKEN_DATA_DIR, exist_ok=True)


TRAIN_BIN = os.path.join(TOKEN_DATA_DIR, "train.bin")
VAL_BIN   = os.path.join(TOKEN_DATA_DIR, "val.bin")


In [ ]:

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

def download_data():
    local_train_paths = []
    print("\nDownloading phase 2 training data.")
    for tf in TRAIN_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            local_train_paths.append(p)
            print(f"Downloaded: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
            
    test_paths = []
    for tf in TEST_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            test_paths.append(p); print(f"Downloaded Test File: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
    return local_train_paths, test_paths

In [ ]:
train_paths, test_paths = download_data()

# inspect_json_files(train_paths, n=2)

In [ ]:
P = {os.path.basename(p): p for p in train_paths}
TEST = test_paths
print(P.keys(), TEST)

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok


In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~149.8M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

In [ ]:
#!cp llama_scratch.py bpe_tokenizer.py phase2_finetune.py

In [ ]:
from huggingface_hub import hf_hub_download
for f in ["vocab.json","merges.json","tokenizer_config.json"]:
    hf_hub_download("DumbLad/legal-bpe-tokenizer-phase1-v2", f, repo_type="model",
                    local_dir="/kaggle/working/tok_phase1", token=HF_TOKEN)

In [ ]:
import shutil, os
os.makedirs("data", exist_ok=True)
for p in list(P.values()) + TEST:
    shutil.copy(p, "data/")
print(sorted(os.listdir("data")))

In [ ]:
%%writefile lorma_hook.py
import math, torch, torch.nn as nn, torch.nn.functional as F

_IDX = {}
def _rot_idx(d, device):                      # idx[i, j] = (j - i) mod d  -> row i rotated right by i
    k = (d, str(device))
    if k not in _IDX:
        ar = torch.arange(d, device=device)
        _IDX[k] = (ar[None, :] - ar[:, None]) % d
    return _IDX[k]

class LoRMALinear(nn.Module):
    def __init__(self, lin: nn.Linear, r, variant, alpha):
        super().__init__()
        assert lin.bias is None                # your Llama layers are bias-free
        self.base, self.variant, self.r = lin, variant, r
        self.scale = alpha / r
        for p in self.base.parameters(): p.requires_grad = False
        d = lin.out_features
        self.A = nn.Parameter(torch.zeros(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":                  # LoRMA+: B = 0, A random
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        else:                                  # LoRMAπ: pi(BA) = I at init
            nn.init.normal_(self.B, std=0.02)
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A.zero_(); self.A[0, 0] = 1.0

    def forward(self, x):
        y0 = self.base(x)                      # W0 x   (multiplication is applied to the output dim)
        if self.variant == "plus":
            return y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        M = self.B @ self.A                    # d x d, rank <= r
        P = M.gather(1, _rot_idx(M.size(0), M.device))   # permutation-based rank inflation
        return F.linear(y0, P)                 # = P @ y0

def apply_lorma(base, r=16, target_modules=("wq","wk","wv","wo","w_gate","w_up","w_down"), variant="plus", alpha=None):
    for p in base.parameters(): p.requires_grad = False
    alpha = alpha or 2 * r
    for _, mod in list(base.named_modules()):
        for cname, child in list(mod.named_children()):
            if cname in target_modules and isinstance(child, nn.Linear):
                setattr(mod, cname, LoRMALinear(child, r, variant, alpha))
    return base

In [ ]:
import torch, torch.nn as nn
from lorma_hook import LoRMALinear
lin = nn.Linear(32, 48, bias=False); x = torch.randn(4, 32)
for v in ("plus", "pi"):
    print(v, torch.allclose(LoRMALinear(lin, 4, v, 8)(x), lin(x), atol=1e-6))   # both True

In [ ]:
%%writefile rr_peft.py
import os, re, json, math, time, random, argparse, copy
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score
from llama_scratch import Config, LlamaLM, PRESETS
from bpe_tokenizer import PureByteLevelBPETokenizer

LABELS = ["Fact","Issue","ArgumentPetitioner","ArgumentRespondent","Statute","Dissent",
          "PrecedentReliedUpon","PrecedentNotReliedUpon","PrecedentOverruled",
          "RulingByLowerCourt","RatioOfTheDecision","RulingByPresentCourt","None"]
L2I = {l: i for i, l in enumerate(LABELS)}; NC = len(LABELS)
ALL_LIN = ["wq","wk","wv","wo","w_gate","w_up","w_down"]

# ---------------- data + preprocessing ----------------
MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*"
def clean(s, pre):
    s = re.sub(r"\s+", " ", s).strip()
    if pre == "raw": return s
    s = re.sub(r"^\(?\d+(\s*\.\s*\d+)*\s*[\.\)]?\s+(?=\D)", "", s)       # leading "2.1 ", "4 .1 ", "3 "
    if pre == "mask":
        s = re.sub(r"\b\d{4}\s+\d+\s+[A-Z][A-Za-z]*\s+\d+\b", "<cite>", s)  # 1979 117 ITR 930
        s = re.sub(r"\b\d{1,2}(?:st|nd|rd|th)?\s+" + MONTH + r",?\s+\d{4}\b", "<date>", s)
        s = re.sub(r"\b\d{1,2}\.\d{1,2}\.\d{2,4}\b", "<date>", s)
        s = re.sub(r"Rs\.?\s*[\d,\.]+(?:\s*(?:crores?|lakhs?))?", "<amt>", s)
        s = re.sub(r"\d{3,}", "<num>", s)
    return s

def load_docs(data_dir, names, frac=1.0, seed=0):
    out = []
    for n in names:
        docs = json.load(open(os.path.join(data_dir, n)))
        for i, d in enumerate(docs): d["id"] = f"{n}:{d.get('id', i)}"
        if frac < 1.0: docs = random.Random(seed).sample(docs, max(1, int(frac * len(docs))))
        out += docs
    return out

def build_examples(docs, c, tok, labeled=True):
    BOS, EOS = tok.bos_token_id, tok.eos_token_id
    k = c["k"] if c["mode"] == "win" else 0
    ex = []
    for d in docs:
        n = len(d["text"])
        cap = 128 if c["mode"] != "win" else max(16, min(128, (c["max_len"] - 2) // (2 * k + 1) - 1))
        S = []
        for i, t in enumerate(d["text"]):
            t = clean(t, c["pre"])
            if c["pos"]: t = f"[{int(10 * i / max(1, n - 1))}] " + t
            ids = tok.encode(t)
            if len(ids) > cap: ids = ids[: cap * 3 // 4] + ids[-(cap - cap * 3 // 4):]
            S.append(ids + [EOS])
        y = [L2I[l] for l in d["labels"]] if labeled else [0] * n
        if c["mode"] == "doc":
            i = 0
            while i < n:
                body, spans, ys, j = [], [], [], i
                lim = (c["max_len"] - 1) // 2 if c.get("echo") else c["max_len"] - 1
                while j < n and len(body) + len(S[j]) <= lim:
                    spans.append((len(body), len(body) + len(S[j]))); body += S[j]; ys.append(y[j]); j += 1
                if c.get("echo"):
                    off = 1 + len(body)                       # start of copy 2 (after BOS + copy 1)
                    seq = [BOS] + body + body
                else:
                    off = 1; seq = [BOS] + body
                spans = [(s + off, e + off) for s, e in spans]
                ex.append(dict(ids=seq, spans=spans, y=ys, keys=[f"{d['id']}|{q}" for q in range(i, j)])); i = j
        else:                                        # sent (k=0) or win: neighbours first, target LAST
            for i in range(n):
                seq = [BOS]
                for j in list(range(max(0, i - k), i)) + list(range(i + 1, min(n, i + 1 + k))): seq += S[j]
                spans = [(len(seq), len(seq) + len(S[i]))]; seq += S[i]
                ex.append(dict(ids=seq, spans=spans, y=[y[i]], keys=[f"{d['id']}|{i}"]))
    return ex

def collate(batch, pad):
    T = max(len(e["ids"]) for e in batch)
    ids = torch.full((len(batch), T), pad, dtype=torch.long)
    bi, st, en, yy, keys = [], [], [], [], []
    for b, e in enumerate(batch):
        ids[b, :len(e["ids"])] = torch.tensor(e["ids"])          # right padding is safe with causal attention
        for (s, t), y, k in zip(e["spans"], e["y"], e["keys"]):
            bi.append(b); st.append(s); en.append(t); yy.append(y); keys.append(k)
    return ids, torch.tensor(bi), torch.tensor(st), torch.tensor(en), torch.tensor(yy), keys

def make_batches(ex, bs, shuffle, rng):
    idx = list(range(len(ex)))
    if shuffle: rng.shuffle(idx)
    out = []
    for i in range(0, len(idx), bs * 50):
        part = sorted(idx[i:i + bs * 50], key=lambda j: len(ex[j]["ids"]))
        out += [part[q:q + bs] for q in range(0, len(part), bs)]
    if shuffle: rng.shuffle(out)
    return out

# ---------------- model ----------------
def load_base(ckpt, keep_every=None, keep_layers=None):
    ck = torch.load(ckpt, map_location="cpu", weights_only=False)
    print("checkpoint top-level keys:", list(ck.keys())[:8], flush=True)
    sd = next((ck[k] for k in ("model", "model_state_dict", "state_dict") if isinstance(ck.get(k), dict)), ck)
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    cfg = ck.get("config") or ck.get("cfg")
    if isinstance(cfg, dict): cfg = Config(**cfg)
    if cfg is None:                                   # raw state_dict: rebuild config from tensor shapes
        V, D = sd["tok_emb.weight"].shape
        L = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("layers."))
        nh = next(p["n_heads"] for p in PRESETS.values() if p["d_model"] == D)
        nkv = sd["layers.0.attn.wk.weight"].shape[0] // (D // nh)
        cfg = Config(vocab_size=V, d_model=D, n_layers=L, n_heads=nh, n_kv_heads=nkv,
                     ffn_hidden=sd["layers.0.mlp.w_gate.weight"].shape[0])
    cfg.attn_impl = "sdpa"
    if keep_every or keep_layers:
        L = cfg.n_layers
        keep = sorted(set(keep_layers)) if keep_layers else sorted(set(range(0, L, keep_every)) | {L - 1})
        remap = {o: n for n, o in enumerate(keep)}; new = {}
        for k, v in sd.items():
            if k.startswith("layers."):
                p = k.split("."); o = int(p[1])
                if o not in remap: continue
                p[1] = str(remap[o]); k = ".".join(p)
            new[k] = v
        sd = new; cfg.n_layers = len(keep); print("student keeps layers", keep, flush=True)
    print("config:", cfg, flush=True)
    m = LlamaLM(cfg)
    res = m.load_state_dict(sd, strict=False)
    bad = [k for k in res.missing_keys if "lm_head" not in k]
    assert not bad and not res.unexpected_keys, (bad[:5], res.unexpected_keys[:5])
    return m

def attach_peft(base, c, total_steps):
    p, r = c["peft"], c["r"]
    if p == "full": return base
    from peft import get_peft_model, LoraConfig, AdaLoraConfig, IA3Config
    if p == "lora":      cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=ALL_LIN)
    elif p == "rslora":  cfg = LoraConfig(r=r, lora_alpha=16 * math.sqrt(r), lora_dropout=0.05, target_modules=ALL_LIN, use_rslora=True)
    elif p == "adalora": cfg = AdaLoraConfig(init_r=int(1.5 * r), target_r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=ALL_LIN,
                                             tinit=int(.1 * total_steps), tfinal=int(.3 * total_steps), deltaT=10, total_step=total_steps)
    elif p == "ia3":     cfg = IA3Config(target_modules=["wk", "wv", "w_down"], feedforward_modules=["w_down"])
    elif p in ("lorma_plus", "lorma_pi"):
        from lorma_hook import apply_lorma
        return apply_lorma(base, r=r, target_modules=ALL_LIN, variant=p.split("_")[1])
    else: raise ValueError(p)
    return get_peft_model(base, cfg)

class RRModel(nn.Module):
    def __init__(self, base, c, total_steps):
        super().__init__()
        self.peft = attach_peft(base, c, total_steps)
        D = base.cfg.d_model; self.ht = c["head"]
        self.attn_q = nn.Linear(D, 1) if self.ht == "attn" else None
        self.cls = (nn.Sequential(nn.Dropout(.1), nn.Linear(D, D), nn.GELU(), nn.Dropout(.1), nn.Linear(D, NC))
                    if self.ht == "mlp" else nn.Sequential(nn.Dropout(.1), nn.Linear(D, NC)))
        self.ctx = nn.GRU(D, D // 2, batch_first=True, bidirectional=True) if c.get("bigru") else None
    def core(self): return self.peft.get_base_model() if hasattr(self.peft, "get_base_model") else self.peft
    def hidden(self, idx):
        b = self.core(); x = b.tok_emb(idx); T = idx.size(1)
        cos = sin = None
        if b.cfg.pos_enc == "rope": cos, sin = b.rope_cos[:T], b.rope_sin[:T]
        for l in b.layers: x, _ = l(x, cos, sin, b.alibi_slopes, None, None, False)
        return b.norm(x)
    def forward(self, ids, bi, st, en):
        h = self.hidden(ids)
        if self.ht in ("last", "mlp"): r = h[bi, en - 1]                       # hidden state at the sentence's </s>
        else:
            pos = torch.arange(h.size(1), device=h.device)[None]
            m = (pos >= st[:, None]) & (pos < en[:, None]); hh = h[bi]
            if self.ht == "mean": r = (hh * m[..., None]).sum(1) / m.sum(1, keepdim=True)
            else:
                a = self.attn_q(hh).squeeze(-1).float().masked_fill(~m, -1e4).softmax(-1)
                r = (a[..., None].to(hh.dtype) * hh).sum(1)

        if self.ctx is not None:
            r = r.float(); N = r.size(0)
            cnt = torch.bincount(bi, minlength=int(bi.max()) + 1)
            p = torch.arange(N, device=r.device) - (cnt.cumsum(0) - cnt)[bi]
            pad = r.new_zeros(len(cnt), int(cnt.max()), r.size(1)); pad[bi, p] = r
            r = r + self.ctx(pad)[0][bi, p]
        return self.cls(r.float())

def make_loss(kind, counts, dev):
    pr = torch.tensor(counts / counts.sum(), dtype=torch.float32, device=dev).clamp_min(1e-6)
    w = pr.pow(-0.5); w = w / (w * pr).sum()
    if kind == "none": return lambda z, y: F.cross_entropy(z, y)
    if kind == "cw":   return lambda z, y: F.cross_entropy(z, y, weight=w)
    if kind == "logit_adj": la = pr.log(); return lambda z, y: F.cross_entropy(z + la, y)   # inference uses raw logits
    if kind == "focal":
        def f(z, y):
            lp = F.log_softmax(z, -1).gather(1, y[:, None]).squeeze(1)
            return (-(w[y]) * (1 - lp.exp()) ** 2 * lp).sum() / w[y].sum()
        return f
    raise ValueError(kind)

@torch.no_grad()
def predict(model, ex, bs, pad):
    model.eval(); Z, Y, K = [], [], []
    for b in make_batches(ex, bs, False, None):
        ids, bi, st, en, y, k = collate([ex[j] for j in b], pad)
        with torch.autocast("cuda", dtype=torch.float16): z = model(ids.cuda(), bi.cuda(), st.cuda(), en.cuda())
        Z.append(z.float().cpu()); Y.append(y); K += k
    return torch.cat(Z), torch.cat(Y), K

def scores(Z, Y):
    p, y = Z.argmax(-1).numpy(), Y.numpy()
    return dict(macro_f1=float(f1_score(y, p, labels=sorted(set(y.tolist())), average="macro", zero_division=0)))

# ---------------- one experiment ----------------
def run(c):
    t0 = time.time(); dev = torch.device("cuda")
    random.seed(c["seed"]); np.random.seed(c["seed"]); torch.manual_seed(c["seed"]); rng = random.Random(c["seed"])
    tok = PureByteLevelBPETokenizer.load(c["tok_dir"]); pad = tok.pad_token_id
    if c["mode"] == "sent": c["k"] = 0
    tr = load_docs(c["data_dir"], ["CL_train.json", "IT_train.json"], c["frac"], c["seed"])
    dv = load_docs(c["data_dir"], ["CL_dev.json", "IT_dev.json"])
    tr_ex, dv_ex = build_examples(tr, c, tok), build_examples(dv, c, tok)
    counts = np.bincount([y for e in tr_ex for y in e["y"]], minlength=NC).astype(np.float64)
    bs = c["bs"] or {"sent": 64, "win": 16, "doc": 2}[c["mode"]]
    steps = len(make_batches(tr_ex, bs, True, random.Random(0))) * c["epochs"]
    # model = RRModel(load_base(c["ckpt"], c.get("keep_every")), c, steps).to(dev)
    model = RRModel(load_base(c["ckpt"], c.get("keep_every"), c.get("keep_layers")), c, steps).to(dev)
    if c["peft"] == "full":
        for p in model.core().parameters(): p.requires_grad = True
    head = (list(model.cls.parameters()) + (list(model.attn_q.parameters()) if model.attn_q else []) + (list(model.ctx.parameters()) if model.ctx is not None else []))
    hid = {id(p) for p in head}
    ad = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([dict(params=ad, lr=c["lr"], weight_decay=0.0), dict(params=head, lr=1e-3, weight_decay=0.01)])
    base_lrs = [g["lr"] for g in opt.param_groups]; warm = max(1, int(.06 * steps))
    lossf = make_loss(c["bal"], counts, dev); scaler = torch.amp.GradScaler()
    TL = torch.load(c["kd_logits"]) if c.get("kd_logits") else None
    best, best_state, step = -1, None, 0
    hist = []
    vhist = []; run_loss = 0.0
    tt = time.time()
    for ep in range(c["epochs"]):
        model.train()
        for b in make_batches(tr_ex, bs, True, rng):
            ids, bi, st, en, y, keys = collate([tr_ex[j] for j in b], pad)
            sc = step / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, steps - warm)))
            for g, l0 in zip(opt.param_groups, base_lrs): g["lr"] = l0 * sc
            with torch.autocast("cuda", dtype=torch.float16): z = model(ids.to(dev), bi.to(dev), st.to(dev), en.to(dev))
            y = y.to(dev); loss = lossf(z.float(), y)
            if TL is not None:                       # knowledge distillation: (1-a)*CE + a*T^2*KL(teacher||student)
                t = torch.stack([TL[k] for k in keys]).to(dev).float(); T_, a = c["kd_T"], c["kd_alpha"]
                loss = (1 - a) * loss + a * T_ * T_ * F.kl_div(F.log_softmax(z.float() / T_, -1), F.softmax(t / T_, -1), reduction="batchmean")
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            if step % 50 == 0 or step == steps:
                el = time.time() - tt
                print(f'[{c["name"]}] step {step}/{steps} ep {ep} loss {loss.item():.4f} '
                      f'elapsed {el/60:.1f}m eta {el/step*(steps-step)/60:.1f}m', flush=True)
            run_loss += float(loss.detach())
            if step % 10 == 0: hist.append((step, run_loss / 10)); run_loss = 0.0
            if c["peft"] == "adalora": model.peft.base_model.update_and_allocate(step)   # (PEFT's orthogonality reg is skipped since we bypass its forward)
            opt.zero_grad(set_to_none=True)
            if c.get("eval_every") and step % c["eval_every"] == 0:
                Zm, Ym, _ = predict(model, dv_ex, 128 if c["mode"] == "sent" else 16, pad); model.train()
                vhist.append((step, float(F.cross_entropy(Zm, Ym)), scores(Zm, Ym)["macro_f1"]))
        Z, Y, _ = predict(model, dv_ex, 128 if c["mode"] == "sent" else 16, pad); s = scores(Z, Y); print(c["name"], "epoch", ep, s["macro_f1"], flush=True)
        vhist.append((step, float(F.cross_entropy(Z, Y)), s["macro_f1"]))
        if s["macro_f1"] > best:
            best = s["macro_f1"]; best_s = dict(s, epoch=ep)
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    os.makedirs(c["out"], exist_ok=True)
    torch.save(dict(cfg=c, state=best_state), f"{c['out']}/{c['name']}.pt")
    json.dump(hist, open(f"{c['out']}/{c['name']}_loss.json", "w"))
    json.dump(vhist, open(f"{c['out']}/{c['name']}_val.json", "w"))
    if c.get("dump_logits"):                         # teacher soft labels for KD, keyed by doc|sentence index
        model.load_state_dict(best_state, strict=False)
        full = build_examples(load_docs(c["data_dir"], ["CL_train.json", "IT_train.json"]), c, tok)
        Z, _, K = predict(model, full, 128 if c["mode"] == "sent" else 16, pad); torch.save({k: z for k, z in zip(K, Z)}, f"{c['out']}/{c['name']}_trainlogits.pt")
    tp = sum(p.numel() for p in ad)
    return dict(name=c["name"], cfg=c, trainable_adapter_params=tp, trainable_pct=100 * tp / sum(p.numel() for p in model.core().parameters()),
                peak_mem_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60, **best_s)

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--jobs"); a = ap.parse_args()
    for c in json.load(open(a.jobs)):
        res = f"{c['out']}/results.jsonl"
        if os.path.exists(res) and any(json.loads(l)["name"] == c["name"] for l in open(res)): continue
        try: r = run(c)
        except Exception:
            import traceback, gc; print(f"[{c['name']}] FAILED", flush=True); traceback.print_exc(); gc.collect(); torch.cuda.empty_cache(); continue
        with open(res, "a") as f: f.write(json.dumps(r) + "\n")
        torch.cuda.empty_cache()

In [ ]:
# launcher cell
import time

import json, copy, subprocess, os
OUT = "/kaggle/working/rr"; os.makedirs(OUT, exist_ok=True)
BASE = dict(ckpt="/kaggle/input/datasets/koooshiieee/phase2v2/model.pt", tok_dir="/kaggle/working/tok_phase1",
            data_dir="./data", out=OUT, peft="lora", lr=3e-4, r=16, mode="sent", k=2, pre="raw", pos=False,
            bal="none", head="last", epochs=2, frac=0.25, max_len=768, bs=None, seed=67, eval_every=50)
def J(name, base, **kw): d = copy.deepcopy(base); d.update(kw); d["name"] = name; return d

def results(): 
    p = f"{OUT}/results.jsonl"; return [json.loads(l) for l in open(p)] if os.path.exists(p) else []


def run_jobs(jobs):
    done = {r["name"] for r in results()}; jobs = [j for j in jobs if j["name"] not in done]; procs = []
    names = {j["name"] for j in jobs}; t0 = time.time()
    for g in range(2):
        mine = jobs[g::2]
        if not mine: continue
        jf = f"{OUT}/jobs_{g}.json"; json.dump(mine, open(jf, "w"))
        procs.append(subprocess.Popen(["python", "-u", "rr_peft.py", "--jobs", jf],
                     env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)), stdout=open(f"{OUT}/log_{g}.txt", "a"), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in procs):
        time.sleep(60)
        n = len([r for r in results() if r["name"] in names])
        print(f"{time.strftime('%H:%M:%S')} | {n}/{len(jobs)} jobs done | {(time.time()-t0)/60:.0f} min elapsed")
        for g in range(len(procs)):
            try: print(f"   GPU{g}:", open(f"{OUT}/log_{g}.txt").read().strip().splitlines()[-1])
            except Exception: pass

    missing = names - {r["name"] for r in results()}
    if missing: print("FAILED / MISSING:", sorted(missing))


def pick(pred): return max([r for r in results() if pred(r["name"])], key=lambda r: r["macro_f1"])

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN); REPO = f"{api.whoami()['name']}/legal-rr-phase3-v2"
api.create_repo(REPO, repo_type="model", private=True, exist_ok=True)
def push(names, folder, extra=()):
    for n in sorted(set(names)):
        for suf in (".pt", "_loss.json", "_val.json") + tuple(extra):
            f = f"{OUT}/{n}{suf}"
            if os.path.exists(f): api.upload_file(path_or_fileobj=f, path_in_repo=f"{folder}{n}{suf}", repo_id=REPO, repo_type="model")
    if os.path.exists(f"{OUT}/results.jsonl"):
        api.upload_file(path_or_fileobj=f"{OUT}/results.jsonl", path_in_repo="results.jsonl", repo_id=REPO, repo_type="model")

In [ ]:
MAIN_OUT = OUT
OUT = "/kaggle/working/rr_smoke"; os.makedirs(OUT, exist_ok=True)
SM = dict(BASE, out=OUT, frac=0.01, epochs=1)
smoke = [J(f"smoke_{m}", SM, peft=m, lr=3e-4) for m in ["lora", "rslora", "adalora", "lorma_plus", "lorma_pi"]] + [
    J("smoke_ia3", SM, peft="ia3", lr=1e-2), J("smoke_win", SM, mode="win"), J("smoke_doc", SM, mode="doc"),
    J("smoke_attn", SM, head="attn"), J("smoke_mlp", SM, head="mlp"), J("smoke_focal", SM, bal="focal"),
    J("smoke_norm", SM, pre="norm"), J("smoke_mask", SM, pre="mask"), J("smoke_pos", SM, pos=True), 
    J("smoke_bigru", SM, mode="doc", bigru=True), J("smoke_echo", SM, mode="doc", echo=True)]
run_jobs(smoke)
for r in results(): print(r["name"], round(r["macro_f1"], 3), f'{r["minutes"]:.1f}min', f'{r["peak_mem_gb"]:.1f}GB')

run_jobs([J("smoke_dump", SM, dump_logits=True), J("smoke_full", SM, peft="full", lr=5e-5), J("smoke_prune", SM, peft="full", lr=5e-5, keep_every=7)])
run_jobs([J("smoke_kd", SM, peft="full", lr=5e-5, keep_every=7, kd_logits=f"{OUT}/smoke_dump_trainlogits.pt", kd_T=2.0, kd_alpha=0.5)])

from rr_peft import load_base; from llama_scratch import count_params
count_params(load_base(BASE["ckpt"], 7))   # expect ~53.7M (14-layer d=896 model)
OUT = MAIN_OUT

In [ ]:
import sys, zipfile, importlib; sys.path.insert(0, ".")
import rr_peft; importlib.reload(rr_peft)
from rr_peft import RRModel, load_base, build_examples, predict, LABELS
from bpe_tokenizer import PureByteLevelBPETokenizer

def load_trained(name, out=None):
    out = out or OUT
    ck = torch.load(f"{out}/{name}.pt", map_location="cpu", weights_only=False); c = ck["cfg"]
    tok = PureByteLevelBPETokenizer.load(c["tok_dir"])
    model = RRModel(load_base(c["ckpt"], c.get("keep_every"), c.get("keep_layers")), c, 100).cuda()
    res = model.load_state_dict(ck["state"], strict=False); assert not res.unexpected_keys, res.unexpected_keys[:5]
    return model, c, tok

def dev_check(name):          # round-trip test: reloaded model must reproduce the logged dev score
    model, c, tok = load_trained(name)
    docs = rr_peft.load_docs("./data", ["CL_dev.json", "IT_dev.json"])
    Z, Y, _ = predict(model, build_examples(docs, c, tok), 128 if c["mode"] == "sent" else 16, tok.pad_token_id)
    print(name, "reloaded:", rr_peft.scores(Z, Y)["macro_f1"], "| logged:", next(r["macro_f1"] for r in results() if r["name"] == name))


def load_test_docs():
    docs = []
    for fi, f in enumerate(["data/CL_test.json", "data/IT_test.json"]):
        raw = json.load(open(f))
        items = list(raw.items()) if isinstance(raw, dict) else [(d.get("id", i), d) for i, d in enumerate(raw)]
        for k, v in items:
            d = {"text": v} if isinstance(v, list) else dict(v)
            d["orig_id"] = str(k); d["id"] = f"{fi}:{k}"
            docs.append(d)
    return docs

def test_logits(name):
    model, c, tok = load_trained(name); docs = load_test_docs()
    Z, _, K = predict(model, build_examples(docs, c, tok, labeled=False), 128 if c["mode"] == "sent" else 16, tok.pad_token_id)
    return docs, dict(zip(K, Z))


def write_submission(names, pred_path=PREDICTION_PATH, zip_path=SUBMISSION_ZIP):
    outs = [test_logits(n) for n in names]; pred = {}
    for d in outs[0][0]:
        labs = [LABELS[int(sum(F.softmax(o[1][f"{d['id']}|{i}"], -1) for o in outs).argmax())] for i in range(len(d["text"]))]
        assert len(labs) == len(d["text"]) and d["orig_id"] not in pred, d["orig_id"]
        pred[d["orig_id"]] = labs
    assert all(l in LABELS for v in pred.values() for l in v)
    json.dump(pred, open(pred_path, "w"), indent=1)
    if zip_path:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z: z.write(pred_path, "prediction.json")
    print(len(pred), "docs,", sum(map(len, pred.values())), "sentences")



def validate_submission(zip_path=SUBMISSION_ZIP):      # re-opens the zip exactly as Codabench will
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        pred = json.loads(z.read("prediction.json"))
    ref = {d["orig_id"]: len(d["text"]) for d in load_test_docs()}
    assert set(pred) == set(ref), (len(pred), len(ref))
    assert all(len(pred[k]) == n for k, n in ref.items()) and all(l in LABELS for v in pred.values() for l in v)
    print("submission valid:", len(pred), "docs")




# dev_check("final_best_s67"); write_submission(["final_best_s67", "final_best_s68"])

In [ ]:
'''OUT = "/kaggle/working/rr_smoke"
try:
    dev_check("smoke_kd"); write_submission(["smoke_kd"]); validate_submission()
finally:
    OUT = MAIN_OUT'''

In [ ]:
'''shutil.rmtree("/kaggle/working/rr_smoke")'''

In [ ]:
FULL = dict(frac=1.0, epochs=3)

METHOD_LR = {"lora":5e-4,"rslora":5e-4,"adalora":5e-4,"lorma_plus":5e-4,"lorma_pi":5e-4,"ia3":5e-2,"full":5e-5}

run_jobs([J("full_lora", BASE, peft="lora", lr=METHOD_LR["lora"], r=32, **FULL)])

In [ ]:
'''
for r in results():
    if r["name"].startswith("s1_"):
        m = r["cfg"]["peft"]
        if m not in bym or r["macro_f1"] > bym[m]["macro_f1"]: bym[m] = r'''


by_arch = {r["cfg"]["peft"]: r for r in results() if r["name"].startswith("full_")}
b1 = pick(lambda n: n == "full_lora")
push(["full_lora"], "ablation/stage1/")

In [ ]:
def stage(tag, prev, variants):
    base = dict(prev["cfg"], dump_logits=False, mode="doc", frac=1.0, epochs=6, max_len=1024, bs=4, eval_every=200)
    names = {prev["name"]}
    run_jobs([J(f"{tag}_{n}", base, **kw) for n, kw in variants.items()])
    return pick(lambda n: n in names or n.startswith(tag + "_"))

b2 = stage("s2", b1, {"doc": dict(), "doc_echo": dict(echo=True), "doc_bigru": dict(bigru=True)})
b3 = stage("s3", b2, {"norm": dict(pre="norm"), "mask": dict(pre="mask"), "pos": dict(pos=True), "norm_pos": dict(pre="norm", pos=True)})
b4 = stage("s4", b3, {"cw": dict(bal="cw"), "logadj": dict(bal="logit_adj"), "focal": dict(bal="focal")})
b5 = stage("s5", b4, {"mlp": dict(head="mlp"), "mean": dict(head="mean")})

FINAL = dict(b5["cfg"], dump_logits=True)      # dump teacher logits for KD
run_jobs([J(f"fin_{m}", FINAL, peft=m, lr=lr, r=32) for m, lr in METHOD_LR.items()])
by_arch = {r["cfg"]["peft"]: r for r in results() if r["name"].startswith("fin_")}
teacher = max(by_arch.values(), key=lambda r: r["macro_f1"])
push([r["name"] for r in by_arch.values()], "ablation/final/")
print({m: round(r["macro_f1"], 4) for m, r in by_arch.items()}, "->", teacher["name"])

'''b2 = stage("s2", b1, {"doc": dict(mode="doc"), "win2": dict(mode="win", k=2)})
b3 = stage("s3", b2, {"norm": dict(pre="norm"), "mask": dict(pre="mask"), "pos": dict(pos=True), "norm_pos": dict(pre="norm", pos=True)})
b4 = stage("s4", b3, {"cw": dict(bal="cw"), "focal": dict(bal="focal"), "logadj": dict(bal="logit_adj")})
b5 = stage("s5", b4, {"mean": dict(head="mean"), "attn": dict(head="attn"), "mlp": dict(head="mlp")})'''

[r["name"] for r in results() if r["name"].startswith(("s2_","s3_","s4_","s5_"))]

In [ ]:
'''os.makedirs(f"{OUT}/preds", exist_ok=True)

for m in by_arch:
    write_submission([f"fin_{m}"], pred_path=f"{OUT}/preds/{m}.json", zip_path=f"{OUT}/submission_{m}.zip")
    validate_submission(f"{OUT}/submission_{m}.zip")'''

In [ ]:
FULL = dict(frac=1.0, epochs=3)

KD = dict(kd_logits=f"{OUT}/{teacher['name']}_trainlogits.pt", kd_T=2.0, kd_alpha=0.5)
SB = dict(teacher["cfg"], dump_logits=False, ckpt=BASE["ckpt"], tok_dir=BASE["tok_dir"], lr=5e-5, peft="full", frac=1.0, epochs=6)
STUDENT     = dict(SB, keep_every=7)
STUDENT_MID = dict(SB, keep_layers=[3, 5, 8, 11])
run_jobs([J("kd_student_ce", SB, keep_every=7), J("kd_student_ce_mid", SB, keep_layers=[3,5,8,11]),
          J("kd_student_kd", SB, keep_every=7, **KD), J("kd_student_kd_mid", SB, keep_layers=[3,5,8,11], **KD)])
push([teacher["name"], "kd_student_ce", "kd_student_ce_mid", "kd_student_kd", "kd_student_kd_mid"], "final/")

In [ ]:
import gc, pandas as pd
SUB = f"{OUT}/submissions"; os.makedirs(f"{SUB}/preds", exist_ok=True)
CHECK = False          # True = also round-trip dev_check every model (adds ~4 min)
ok, bad = [], []
for r in results():
    n = r["name"]
    if not os.path.exists(f"{OUT}/{n}.pt"):
        bad.append((n, "no checkpoint")); continue
    try:
        if CHECK: dev_check(n)
        write_submission([n], pred_path=f"{SUB}/preds/{n}.json", zip_path=f"{SUB}/{n}.zip")
        validate_submission(f"{SUB}/{n}.zip"); ok.append(n)
    except Exception as e:
        bad.append((n, repr(e)[:150]))
    gc.collect(); torch.cuda.empty_cache()

idx = pd.DataFrame([dict(run=r["name"], dev_macro_f1=round(r["macro_f1"], 4), best_epoch=r["epoch"],
                         zip=f"{SUB}/{r['name']}.zip") for r in results() if r["name"] in ok])
idx.sort_values("dev_macro_f1", ascending=False).to_csv(f"{SUB}/index.csv", index=False)
print(len(ok), "submissions written | FAILED:", bad); idx.sort_values("dev_macro_f1", ascending=False)

In [ ]:
# teacher = final_best_s67 (its train-set logits were dumped above)

# KD = dict(kd_logits=f"{OUT}/final_best_s67_trainlogits.pt", kd_T=2.0, kd_alpha=0.5)
'''run_jobs([J("kd_student_kd", b5["cfg"], **STUDENT, **KD),
          J("kd_student_kd_mid", b5["cfg"], **STUDENT_MID, **KD)])'''
# push(["kd_student_kd", "kd_student_kd_mid"], "kd/")

In [ ]:
dev_check("kd_student_kd"); write_submission(["kd_student_kd"]); validate_submission()

In [ ]:
'''os.makedirs(f"{OUT}/preds", exist_ok=True)
for tag, nm in [(m, r["name"]) for m, r in by_arch.items()] + [("teacher", "final_best_s67"), ("student_ce", "kd_student_ce"), ("student_kd", "kd_student_kd")]:
    dev_check(nm)
    write_submission([nm], pred_path=f"{OUT}/preds/{tag}.json", zip_path=None)'''

In [ ]:
import pandas as pd
cols = ["name", "macro_f1", "trainable_pct", "peak_mem_gb", "minutes", "epoch"]
df = pd.DataFrame([{k: r.get(k) for k in cols} for r in results()]).sort_values("name")
df.to_csv(f"{OUT}/results.csv", index=False); df

In [ ]:
import matplotlib.pyplot as plt, numpy as np
def smooth(y, k=5):
    y = np.asarray(y, float)
    return y if len(y) < k else np.convolve(np.pad(y, (k-1, 0), mode="edge"), np.ones(k)/k, mode="valid")

def plot(names, path, title, k=5):
    cols = plt.cm.tab20(np.linspace(0, 1, 20)); ls = ["-", "--", ":"]
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    for i, n in enumerate(names):
        st = dict(color=cols[i % 20], ls=ls[(i // 20) % 3], label=n, lw=1.4)
        h = json.load(open(f"{OUT}/{n}_loss.json"))
        ax[0].plot([a for a, _ in h], smooth([b for _, b in h], k), **st)
        vp = f"{OUT}/{n}_val.json"
        if os.path.exists(vp):
            v = json.load(open(vp))
            ax[1].plot([a for a, _, _ in v], [b for _, b, _ in v], marker="o", **st)
            ax[2].plot([a for a, _, _ in v], [c for _, _, c in v], marker="o", **st)
    for a, t in zip(ax, ["train loss (moving avg)", "dev cross-entropy (unweighted)", "dev macro-F1"]):
        a.set_title(t); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0].legend(fontsize=6); fig.suptitle(title); plt.tight_layout()
    plt.savefig(path, dpi=150); plt.show(); plt.close()

In [ ]:
stage_names = [r["name"] for r in results() if r["name"].startswith(("s2_","s3_","s4_","s5_"))]

plot([r["name"] for r in by_arch.values()], f"{OUT}/loss_best_per_arch.png", "PEFT methods, final config")
plot(stage_names, f"{OUT}/loss_stages.png", "Stages 2-5")
plot([teacher["name"], "kd_student_ce", "kd_student_ce_mid", "kd_student_kd", "kd_student_kd_mid"],
     f"{OUT}/loss_final_kd.png", "Teacher vs students")

In [ ]:
from llama_scratch import count_params
n_t = count_params(load_base(BASE["ckpt"]), verbose=False) / 1e6
n_s = count_params(load_base(BASE["ckpt"], 7), verbose=False) / 1e6

def md(rows, cols):
    f = lambda v: f"{v:.4g}" if isinstance(v, float) else str(v)
    return "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] + ["| " + " | ".join(f(r.get(c, "")) for c in cols) + " |" for r in rows])

def row(r): c = r["cfg"]; return dict(run=r["name"], method=c["peft"], lr=c["lr"], rank=c["r"], dev_macro_f1=r["macro_f1"],
                                       best_epoch=r["epoch"], trainable_pct=round(r["trainable_pct"], 2), minutes=round(r["minutes"], 1))
R = {r["name"]: r for r in results()}
arch_rows = [row(r) for _, r in sorted(by_arch.items(), key=lambda kv: -kv[1]["macro_f1"])]
final_rows = [row(R[n]) for n in (teacher["name"], "kd_student_ce", "kd_student_ce_mid", "kd_student_kd", "kd_student_kd_mid") if n in R]
stage_rows = [row(R[n]) for n in sorted(R) if n.startswith(("s2_", "s3_", "s4_", "s5_"))]
sel = {k: b5["cfg"].get(k) for k in ("peft", "lr", "r", "mode", "pre", "pos", "bal", "head", "bigru", "echo")}
cols = ["run", "method", "lr", "rank", "dev_macro_f1", "best_epoch", "trainable_pct", "minutes"]

readme = "\n".join([
"---\ntags: [legal-nlp, rhetorical-roles, sequence-classification, peft, knowledge-distillation]\n---",
"# Legal Rhetorical Role Classification (CS-781 Capstone, Phase 3)",
"Sentence-level rhetorical-role labelling of Indian court judgments (13 labels). A Llama-style LM written from scratch and pretrained in Phase 1 is "
"fine-tuned with several parameter-efficient methods, the best recipe is selected by ablation on the dev set, and a ~50M-parameter student is distilled from it.",
"**Primary metric: macro-F1** (labels present in the evaluation set, pooled over all sentences).",
"## Labels\n" + ", ".join(LABELS),
"## Base model (Phase 1)\n" + f"Decoder-only Llama-style LM: RMSNorm, ALiBi, grouped-query attention (12 heads, 4 KV heads), SwiGLU, tied embeddings, 33k byte-level BPE vocabulary, d_model 768, 12 layers (~{n_t:.1f}M params). "
"Everything (model, tokenizer, training loop) is implemented from scratch in `code/llama_scratch.py` and `code/bpe_tokenizer.py`. The Phase 1 checkpoint is **not** in this repo; adapter checkpoints need it as the frozen base.",
"## Method\n"
"- **Classifier:** the LM runs over the sentence (`<s> sentence </s>`); the hidden state at `</s>` (or mean / attention pooling / MLP head, see ablation) feeds a linear classifier. Sentences longer than 128 tokens keep their head and tail.\n"
"- **Fine-tuning methods compared:** LoRA, rsLoRA, AdaLoRA, IA3, LoRMA+ and LoRMA-pi (`code/lorma_hook.py`, multiplicative low-rank adaptation with permutation-based rank inflation), all on the 7 linear projections (wq, wk, wv, wo, gate, up, down) unless noted.\n"
"- **Context modes:** `sent` (one sentence), `win` (k neighbours either side), `doc` (consecutive sentences packed, each classified at its own `</s>`).\n"
"- **Preprocessing:** `raw`, `norm` (strip leading numbering), `mask` (also mask citations, dates, amounts, long numbers); optional relative-position prefix.\n"
"- **Imbalance handling:** none, class-weighted CE (p^-0.5), focal, logit adjustment.\n"
"- **Optimiser:** AdamW, cosine schedule with 6% warmup, fp16 autocast, grad-clip 1.0, head LR 1e-3. Best epoch chosen on dev macro-F1.",
"## Ablation protocol\n"
"1. **Baseline:** LoRA, sentence-only, 100% of the training data, 3 epochs.\n"
"2. **Stages 2-5 (greedy, cumulative, LoRA, 100% data, 6 epochs):** context (doc / BiGRU / echo), preprocessing, imbalance loss, pooling head. Each stage keeps the best config so far.\n"
"3. **PEFT ablation:** all 7 methods trained in the selected configuration.\n"
"4. **Distillation:** teacher = best method from step 3; student = Phase-1 weights with layers pruned to [0, 7, 11] "
f"~{n_s:.1f}M parameters, fully fine-tuned with loss 0.5*CE + 0.5*T^2*KL (T=2), compared against the same student trained with CE only.",
"## Results (dev macro-F1)\n### Fine-tuning methods in the selected configuration (100% data)" + md(arch_rows, cols),
"### Greedy stages 2-5\n" + md(stage_rows, cols),
"### Final models (100% training data)\n" + md(final_rows, cols),
"Selected configuration: `" + json.dumps(sel) + "`",
"## Repository layout\n"
"- `ablation/stage1/`, `ablation/stages/`, `final/`, `kd/`: checkpoints (`.pt` = trainable weights of the best epoch, `_loss.json` = training loss) per run\n"
"- `results.jsonl`, `results.csv`: every run's config and dev scores\n"
"- `preds/`: test-set predictions of the best run of each method, teacher and both students\n"
"- `prediction.json`, `submission.zip`: the Codabench submission (KD student)\n"
"- `tokenizer/`: Phase 1 BPE tokenizer; `code/`: all source files; `*.png`: training-loss curves",
"## Submission format\n"
'`prediction.json` maps each test document id to an ordered list of labels, one per sentence, e.g. `{"document_001": ["Fact", "Issue", ...]}`. '
"Lengths are asserted to equal the number of sentences per document and re-validated from the zip.",
"## Reproduce\nRun the notebook top to bottom on a Kaggle 2xT4 session (jobs are resumable; finished runs are skipped).",
"## Limitations\n- Design choices were tuned with LoRA only.\n"
"- Dev set is used for model and epoch selection, so dev scores are optimistic.\n- Single seed per run.",
"## References\nLoRA; rsLoRA; AdaLoRA; IA3; LoRMA; Training-free layer selection for partial fine-tuning of language models; Hinton et al., knowledge distillation.",
])
open(f"{OUT}/README.md", "w").write(readme)

up = lambda p, dst: api.upload_file(path_or_fileobj=p, path_in_repo=dst, repo_id=REPO, repo_type="model")
up(f"{OUT}/README.md", "README.md")
for f in ["loss_best_per_arch.png", "loss_stages.png", "loss_final_kd.png", "results.csv"]: up(f"{OUT}/{f}", f)
for f in ["rr_peft.py", "llama_scratch.py", "bpe_tokenizer.py", "lorma_hook.py"]: up(f, f"code/{f}")
for f in ["vocab.json", "merges.json", "tokenizer_config.json"]: up(f"/kaggle/working/tok_phase1/{f}", f"tokenizer/{f}")
for f in [PREDICTION_PATH, SUBMISSION_ZIP]: up(f, os.path.basename(f))

api.upload_folder(folder_path=SUB, path_in_repo="submissions", repo_id=REPO, repo_type="model")
print("Uploaded to https://huggingface.co/" + REPO)